In [1]:
import numpy as np
import pandas as pd
from math import log

np.random.seed(42)

# ============================================================
# BUILD AND TRAIN A PROBABILISTIC MODEL USING MLE AND EM
# Example domain:
# Email classification with observed word indicators and a hidden topic
#
# Observed variables:
#   X1 = contains the word "offer"       (0/1)
#   X2 = contains the word "meeting"     (0/1)
#
# Hidden variable:
#   Z = latent email type
#       0 -> promotional
#       1 -> work
#
# Model:
#   P(Z) * P(X1 | Z) * P(X2 | Z)
# ============================================================

# ------------------------------------------------------------
# Synthetic observed dataset
# Each row is an email with two observed binary features
# ------------------------------------------------------------

N = 300

true_pi = np.array([0.55, 0.45])  # P(Z=0), P(Z=1)

# Bernoulli parameters
# P(X1=1 | Z), P(X2=1 | Z)
true_theta = {
    0: {"offer": 0.80, "meeting": 0.15},  # promotional emails
    1: {"offer": 0.20, "meeting": 0.75},  # work emails
}

Z_true = np.random.choice([0, 1], size=N, p=true_pi)
X = []

for z in Z_true:
    x1 = np.random.binomial(1, true_theta[z]["offer"])
    x2 = np.random.binomial(1, true_theta[z]["meeting"])
    X.append([x1, x2])

X = np.array(X)
df = pd.DataFrame(X, columns=["offer", "meeting"])

print("=" * 80)
print("PROBABILISTIC MODEL WITH MLE AND EM")
print("=" * 80)

print("\nObserved dataset preview:")
print(df.head())

# ------------------------------------------------------------
# Part 1: Statistical Learning (20.1)
# Define a probability distribution and compute MLE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 1: STATISTICAL LEARNING (20.1)")
print("=" * 80)

print("\nWe first define a simple observed-data probability model for one binary variable:")
print("Y = whether an email contains the word 'offer'")
print("Assume Y ~ Bernoulli(p)")

Y = df["offer"].values
mle_p = Y.mean()

def bernoulli_log_likelihood(data, p):
    eps = 1e-12
    p = min(max(p, eps), 1 - eps)
    return np.sum(data * np.log(p) + (1 - data) * np.log(1 - p))

ll_mle = bernoulli_log_likelihood(Y, mle_p)
ll_bad = bernoulli_log_likelihood(Y, 0.50)

print(f"\nMLE for p = P(offer=1): {mle_p:.4f}")
print(f"Log-likelihood at MLE: {ll_mle:.4f}")
print(f"Log-likelihood at p=0.50: {ll_bad:.4f}")

print("\nInterpretation:")
print("- The Bernoulli MLE is the sample mean.")
print("- This parameter maximizes the likelihood of the observed binary data under the model.")

# ------------------------------------------------------------
# Part 2: Complete Data Learning (20.2)
# Learn conditional probabilities from complete data
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 2: COMPLETE DATA LEARNING (20.2)")
print("=" * 80)

print("\nNow suppose the latent class Z were actually observed.")
print("Then we could estimate the full conditional model directly from complete data.")

complete_df = pd.DataFrame({
    "offer": X[:, 0],
    "meeting": X[:, 1],
    "latent_type": Z_true
})

# MLE for complete-data parameters
count_z0 = np.sum(Z_true == 0)
count_z1 = np.sum(Z_true == 1)

pi_hat_complete = np.array([
    count_z0 / N,
    count_z1 / N
])

theta_hat_complete = {
    0: {
        "offer": complete_df.loc[complete_df["latent_type"] == 0, "offer"].mean(),
        "meeting": complete_df.loc[complete_df["latent_type"] == 0, "meeting"].mean()
    },
    1: {
        "offer": complete_df.loc[complete_df["latent_type"] == 1, "offer"].mean(),
        "meeting": complete_df.loc[complete_df["latent_type"] == 1, "meeting"].mean()
    }
}

print("\nComplete-data MLE estimates:")
print(f"P(Z=0) = {pi_hat_complete[0]:.4f}")
print(f"P(Z=1) = {pi_hat_complete[1]:.4f}")
print(f"P(offer=1 | Z=0) = {theta_hat_complete[0]['offer']:.4f}")
print(f"P(meeting=1 | Z=0) = {theta_hat_complete[0]['meeting']:.4f}")
print(f"P(offer=1 | Z=1) = {theta_hat_complete[1]['offer']:.4f}")
print(f"P(meeting=1 | Z=1) = {theta_hat_complete[1]['meeting']:.4f}")

def infer_posterior_z(x1, x2, pi, theta):
    scores = []
    for z in [0, 1]:
        pz = pi[z]
        p1 = theta[z]["offer"] if x1 == 1 else (1 - theta[z]["offer"])
        p2 = theta[z]["meeting"] if x2 == 1 else (1 - theta[z]["meeting"])
        scores.append(pz * p1 * p2)
    scores = np.array(scores)
    return scores / scores.sum()

example = (1, 0)
post_complete = infer_posterior_z(example[0], example[1], pi_hat_complete, theta_hat_complete)
print(f"\nInference example for email features offer={example[0]}, meeting={example[1]}:")
print(f"P(Z=0 | x) = {post_complete[0]:.4f}")
print(f"P(Z=1 | x) = {post_complete[1]:.4f}")

# ------------------------------------------------------------
# Part 3: Hidden Variables and EM (20.3)
# Introduce latent variables and estimate with EM
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 3: HIDDEN VARIABLES AND EM (20.3)")
print("=" * 80)

print("\nNow hide Z and learn from observed data only.")
print("We use EM for a latent-variable Bernoulli mixture model.")

# Random-ish initialization
pi_em = np.array([0.50, 0.50])
theta_em = {
    0: {"offer": 0.60, "meeting": 0.40},
    1: {"offer": 0.40, "meeting": 0.60}
}

def log_likelihood_observed(X, pi, theta):
    total = 0.0
    eps = 1e-12
    for x1, x2 in X:
        px = 0.0
        for z in [0, 1]:
            pz = pi[z]
            p1 = theta[z]["offer"] if x1 == 1 else (1 - theta[z]["offer"])
            p2 = theta[z]["meeting"] if x2 == 1 else (1 - theta[z]["meeting"])
            px += pz * p1 * p2
        total += np.log(max(px, eps))
    return total

def e_step(X, pi, theta):
    responsibilities = np.zeros((len(X), 2))
    for i, (x1, x2) in enumerate(X):
        for z in [0, 1]:
            pz = pi[z]
            p1 = theta[z]["offer"] if x1 == 1 else (1 - theta[z]["offer"])
            p2 = theta[z]["meeting"] if x2 == 1 else (1 - theta[z]["meeting"])
            responsibilities[i, z] = pz * p1 * p2
        responsibilities[i, :] /= responsibilities[i, :].sum()
    return responsibilities

def m_step(X, responsibilities):
    Nk = responsibilities.sum(axis=0)
    pi_new = Nk / len(X)

    theta_new = {}
    for z in [0, 1]:
        gamma_z = responsibilities[:, z]
        theta_new[z] = {
            "offer": np.sum(gamma_z * X[:, 0]) / Nk[z],
            "meeting": np.sum(gamma_z * X[:, 1]) / Nk[z]
        }
    return pi_new, theta_new

max_iters = 40
tol = 1e-6
history = []

prev_ll = None

for iteration in range(1, max_iters + 1):
    # E-step
    responsibilities = e_step(X, pi_em, theta_em)

    # M-step
    pi_em, theta_em = m_step(X, responsibilities)

    # Evaluate observed-data log-likelihood
    ll = log_likelihood_observed(X, pi_em, theta_em)
    history.append(ll)

    print(f"\nIteration {iteration}")
    print(f"Log-likelihood: {ll:.6f}")
    print(f"pi = [{pi_em[0]:.4f}, {pi_em[1]:.4f}]")
    print(f"P(offer=1 | Z=0) = {theta_em[0]['offer']:.4f}")
    print(f"P(meeting=1 | Z=0) = {theta_em[0]['meeting']:.4f}")
    print(f"P(offer=1 | Z=1) = {theta_em[1]['offer']:.4f}")
    print(f"P(meeting=1 | Z=1) = {theta_em[1]['meeting']:.4f}")

    if prev_ll is not None and abs(ll - prev_ll) < tol:
        print("Converged.")
        break
    prev_ll = ll

print("\nFinal EM estimates:")
print(f"P(Z=0) = {pi_em[0]:.4f}")
print(f"P(Z=1) = {pi_em[1]:.4f}")
print(f"P(offer=1 | Z=0) = {theta_em[0]['offer']:.4f}")
print(f"P(meeting=1 | Z=0) = {theta_em[0]['meeting']:.4f}")
print(f"P(offer=1 | Z=1) = {theta_em[1]['offer']:.4f}")
print(f"P(meeting=1 | Z=1) = {theta_em[1]['meeting']:.4f}")

print("\nObserved-data log-likelihood progression:")
for i, ll in enumerate(history, start=1):
    print(f"Iteration {i:2d}: {ll:.6f}")

# Inference with learned EM model
example2 = (1, 1)
post_em = infer_posterior_z(example2[0], example2[1], pi_em, theta_em)
print(f"\nPosterior inference using EM parameters for email offer={example2[0]}, meeting={example2[1]}:")
print(f"P(Z=0 | x) = {post_em[0]:.4f}")
print(f"P(Z=1 | x) = {post_em[1]:.4f}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("- Part 1 used maximum likelihood on a simple Bernoulli model.")
print("- Part 2 estimated conditional probabilities when the latent class was observed.")
print("- Part 3 hid the class variable and used EM to estimate the model.")
print("- The log-likelihood increased across iterations, showing EM convergence behavior.")
print("=" * 80)

PROBABILISTIC MODEL WITH MLE AND EM

Observed dataset preview:
   offer  meeting
0      1        0
1      0        1
2      0        0
3      0        1
4      1        0

PART 1: STATISTICAL LEARNING (20.1)

We first define a simple observed-data probability model for one binary variable:
Y = whether an email contains the word 'offer'
Assume Y ~ Bernoulli(p)

MLE for p = P(offer=1): 0.5667
Log-likelihood at MLE: -205.2695
Log-likelihood at p=0.50: -207.9442

Interpretation:
- The Bernoulli MLE is the sample mean.
- This parameter maximizes the likelihood of the observed binary data under the model.

PART 2: COMPLETE DATA LEARNING (20.2)

Now suppose the latent class Z were actually observed.
Then we could estimate the full conditional model directly from complete data.

Complete-data MLE estimates:
P(Z=0) = 0.5500
P(Z=1) = 0.4500
P(offer=1 | Z=0) = 0.8667
P(meeting=1 | Z=0) = 0.1515
P(offer=1 | Z=1) = 0.2000
P(meeting=1 | Z=1) = 0.7037

Inference example for email features offer=1, me

# Probabilistic Model with Maximum Likelihood and EM

## Model design

This notebook builds a probabilistic model for classifying emails based on two observed binary features:

- `offer`: whether the email contains the word “offer”
- `meeting`: whether the email contains the word “meeting”

The model also includes a hidden variable:

- `Z`: latent email type  
  - `Z = 0` means promotional
  - `Z = 1` means work-related

The full probabilistic model is:

\[
P(Z) \cdot P(\text{offer} \mid Z) \cdot P(\text{meeting} \mid Z)
\]

This is a simple latent-variable generative model with conditional independence between observed features given the hidden class.

---

## Part 1: Statistical Learning (20.1)

### Probability distribution
The first step uses a simpler observed-data model with one binary variable:

\[
Y \sim \text{Bernoulli}(p)
\]

where \(Y = 1\) means the word “offer” appears in the email.

### Maximum likelihood estimate
For Bernoulli data, the maximum likelihood estimate is:

\[
\hat{p} = \frac{1}{N}\sum_{i=1}^{N} y_i
\]

So the MLE is simply the sample mean.

### Likelihood of data
The likelihood for Bernoulli observations is:

\[
L(p) = \prod_{i=1}^{N} p^{y_i}(1-p)^{1-y_i}
\]

and the log-likelihood is:

\[
\log L(p) = \sum_{i=1}^{N} \left[y_i \log p + (1-y_i)\log(1-p)\right]
\]

The code computes this value and compares the log-likelihood at the MLE to another candidate parameter value.

---

## Part 2: Complete Data Learning (20.2)

### Learning conditional probabilities from data
In this part, the hidden variable \(Z\) is assumed to be observed.

That means the complete-data model can be estimated directly:

- \(P(Z)\)
- \(P(\text{offer}=1 \mid Z)\)
- \(P(\text{meeting}=1 \mid Z)\)

These are learned using frequency estimates from the complete data.

### Parameter estimates
For example:

\[
\hat{P}(Z=z) = \frac{\text{count}(Z=z)}{N}
\]

\[
\hat{P}(\text{offer}=1 \mid Z=z) = \frac{\text{count}(\text{offer}=1, Z=z)}{\text{count}(Z=z)}
\]

and similarly for `meeting`.

### Inference using the learned model
Once the parameters are learned, inference can be performed with Bayes’ rule:

\[
P(Z \mid x_1, x_2) \propto P(Z)\,P(x_1 \mid Z)\,P(x_2 \mid Z)
\]

This gives posterior probabilities over the latent class for a new observed email.

---

## Part 3: Hidden Variables and EM (20.3)

### Hidden variables
Now the variable \(Z\) is no longer observed.

So we only see the observed data:

- `offer`
- `meeting`

but not the underlying email type.

This makes direct learning harder because the class memberships are missing.

---

## Likelihood function

With hidden variables, the observed-data likelihood becomes:

\[
L(\theta) = \prod_{i=1}^{N} \sum_{z \in \{0,1\}} P(z)\,P(x_{i1}\mid z)\,P(x_{i2}\mid z)
\]

and the log-likelihood is:

\[
\log L(\theta) = \sum_{i=1}^{N} \log \left( \sum_{z \in \{0,1\}} P(z)\,P(x_{i1}\mid z)\,P(x_{i2}\mid z) \right)
\]

The difficulty is that the sum over latent values is inside the logarithm, so direct maximization is no longer simple.

---

## EM algorithm

To handle hidden variables, the notebook implements the **Expectation-Maximization (EM)** algorithm.

### E-step
In the E-step, the model computes the posterior probability that each example belongs to each latent class:

\[
\gamma_{i,z} = P(Z=z \mid x_i)
\]

These are called **responsibilities**.

### M-step
In the M-step, the parameters are re-estimated using these soft assignments:

\[
\hat{P}(Z=z) = \frac{1}{N}\sum_{i=1}^{N} \gamma_{i,z}
\]

\[
\hat{P}(\text{offer}=1 \mid Z=z) =
\frac{\sum_{i=1}^{N} \gamma_{i,z}\,x_{i,\text{offer}}}{\sum_{i=1}^{N} \gamma_{i,z}}
\]

\[
\hat{P}(\text{meeting}=1 \mid Z=z) =
\frac{\sum_{i=1}^{N} \gamma_{i,z}\,x_{i,\text{meeting}}}{\sum_{i=1}^{N} \gamma_{i,z}}
\]

So EM alternates between estimating latent memberships and updating parameters.

---

## Parameter updates

The parameter updates used in the code are:

### Mixing proportions
\[
\pi_z = \frac{N_z}{N}
\]
where
\[
N_z = \sum_{i=1}^{N} \gamma_{i,z}
\]

### Bernoulli conditional probabilities
\[
\theta_{z,\text{offer}} = \frac{\sum_i \gamma_{i,z} x_{i,\text{offer}}}{N_z}
\]

\[
\theta_{z,\text{meeting}} = \frac{\sum_i \gamma_{i,z} x_{i,\text{meeting}}}{N_z}
\]

These are weighted averages using posterior responsibilities instead of hard labels.

---

## Convergence analysis

The code evaluates the observed-data log-likelihood after each EM iteration.

A key property of EM is that the log-likelihood should not decrease across iterations.  
This means EM monotonically improves or maintains the current solution at each step.

The convergence behavior is shown by printing the log-likelihood history over time.

Typical behavior is:

- large improvements early
- smaller improvements later
- eventual convergence when updates become very small

This indicates that the parameters are stabilizing.

---

## What the notebook demonstrates

This notebook demonstrates all required components:

### Part 1
- defines a probability distribution
- computes MLE
- evaluates likelihood

### Part 2
- learns conditional probabilities from complete data
- performs posterior inference

### Part 3
- introduces latent variables
- implements EM
- shows convergence through likelihood improvement

Overall, the notebook shows how probabilistic learning changes when hidden variables are present and how EM provides a practical method for parameter estimation in that setting.

## Reflection Questions

### How did hidden variables change learning?
Hidden variables changed learning by making the class membership unknown for each training example. When the latent variable was observed, parameter estimation was straightforward because the data could be separated directly by class and conditional probabilities could be computed from counts. Once the hidden variable was removed, the model could no longer assign each example to a class with certainty. Instead, it had to estimate soft probabilities of belonging to each latent group, which made learning iterative rather than direct.

---

### Why can’t we directly maximize likelihood with missing data?
We cannot directly maximize likelihood with missing data because the observed-data likelihood includes a sum over all possible values of the hidden variables inside the logarithm. This makes the objective much harder to optimize than the complete-data case. With complete data, the likelihood factorizes cleanly and frequency-based estimates are available in closed form. With missing latent structure, we must account for uncertainty about unobserved assignments, which prevents a simple direct maximization.

---

### How does EM guarantee improvement each iteration?
EM guarantees improvement each iteration by alternating between two steps. In the E-step, it computes the expected latent assignments under the current parameters. In the M-step, it chooses new parameters that maximize the expected complete-data log-likelihood under those assignments. This creates a lower bound on the observed-data log-likelihood and improves or maintains that bound at every iteration. As a result, the observed-data likelihood is guaranteed not to decrease.

---

### When might EM converge to a local optimum?
EM can converge to a local optimum when the likelihood surface has multiple peaks. Because EM is iterative and depends on its starting values, different initializations can lead to different final solutions. This is especially common in mixture models or more complex latent-variable models where several parameter settings explain the data reasonably well. In practice, EM may converge to a local optimum rather than the global optimum if the initialization is poor or the model is highly non-convex.